In [1]:
import pandas as pd

df = pd.read_parquet(
    r"C:\Users\Lenovo\Downloads\yellow_tripdata_2025-01.parquet"
)

print(df.shape)

(3475226, 20)


In [2]:
print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [3]:
investigation_df = df[
    [
        'VendorID',
        'tpep_pickup_datetime',
        'tpep_dropoff_datetime',
        'passenger_count',
        'trip_distance',
        'PULocationID',
        'DOLocationID',
        'payment_type',
        'fare_amount',
        'tip_amount',
        'total_amount'
    ]
].copy()

print(investigation_df.shape)

(3475226, 11)


In [4]:
investigation_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,229,237,1,10.0,3.00,18.00
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,236,237,1,5.1,2.02,12.12
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,141,141,1,5.1,2.00,12.10
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,244,244,2,7.2,0.00,9.70
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,244,116,2,5.8,0.00,8.30


In [5]:
investigation_df['trip_duration_min'] = (
    investigation_df['tpep_dropoff_datetime']
    - investigation_df['tpep_pickup_datetime']
).dt.total_seconds() / 60

investigation_df[['trip_duration_min', 'trip_distance', 'fare_amount', 'total_amount']].describe()

,trip_duration_min,trip_distance,fare_amount,total_amount
count,3.475226e+06,3.475226e+06,3.475226e+06,3.475226e+06
mean,1.501812e+01,5.855126e+00,1.708180e+01,2.561129e+01
std,3.871358e+01,5.646016e+02,4.634729e+02,4.636585e+02
min,-5.147232e+04,0.000000e+00,-9.000000e+02,-9.010000e+02
25%,7.283333e+00,9.800000e-01,8.600000e+00,1.520000e+01
50%,1.170000e+01,1.670000e+00,1.211000e+01,1.995000e+01
75%,1.833333e+01,3.100000e+00,1.950000e+01,2.778000e+01
max,5.626317e+03,2.764236e+05,8.633721e+05,8.633804e+05


In [6]:
print("Negative duration:", (investigation_df['trip_duration_min'] < 0).sum())
print("Zero distance:", (investigation_df['trip_distance'] == 0).sum())
print("Negative fare:", (investigation_df['fare_amount'] < 0).sum())
print("Negative total:", (investigation_df['total_amount'] < 0).sum())

Negative duration: 124
Zero distance: 90893
Negative fare: 144118
Negative total: 63037


In [7]:
investigation_df['investigation_flag'] = (
    (investigation_df['trip_duration_min'] < 0) |
    (investigation_df['trip_distance'] <= 0) |
    (investigation_df['fare_amount'] < 0) |
    (investigation_df['total_amount'] < 0)
)

print(investigation_df['investigation_flag'].value_counts())

investigation_flag
False    3254338
True      220888
Name: count, dtype: int64


In [8]:
investigation_df['risk_score'] = (
    (investigation_df['trip_duration_min'] < 0).astype(int) +
    (investigation_df['trip_distance'] <= 0).astype(int) +
    (investigation_df['fare_amount'] < 0).astype(int) +
    (investigation_df['total_amount'] < 0).astype(int)
)

print(investigation_df['risk_score'].value_counts().sort_index())

risk_score
0    3254338
1     148013
2      68466
3       4409
Name: count, dtype: int64


In [9]:
def classify_record(row):
    if row['risk_score'] == 0:
        return 'Normal'
    elif row['risk_score'] == 1:
        return 'Single Exception'
    else:
        return 'Multiple Exceptions'

investigation_df['investigation_category'] = investigation_df.apply(
    classify_record, axis=1
)

print(investigation_df['investigation_category'].value_counts())

investigation_category
Normal                 3254338
Single Exception        148013
Multiple Exceptions      72875
Name: count, dtype: int64


In [10]:
print("Zero passengers:", (investigation_df['passenger_count'] == 0).sum())
print("Missing passengers:", investigation_df['passenger_count'].isna().sum())
print("Passengers > 6:", (investigation_df['passenger_count'] > 6).sum())

Zero passengers: 24656
Missing passengers: 540149
Passengers > 6: 18


In [11]:
print(investigation_df.isna().sum())

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count           540149
trip_distance                  0
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
tip_amount                     0
total_amount                   0
trip_duration_min              0
investigation_flag             0
risk_score                     0
investigation_category         0
dtype: int64


In [12]:
clean_df = investigation_df[
    (investigation_df['trip_duration_min'] >= 0) &
    (investigation_df['fare_amount'] >= 0) &
    (investigation_df['total_amount'] >= 0)
].copy()

print("Original records:", len(investigation_df))
print("Clean records:", len(clean_df))
print("Removed records:", len(investigation_df) - len(clean_df))

Original records: 3475226
Clean records: 3330663
Removed records: 144563


In [17]:
clean_df['avg_speed_mph'] = (
    clean_df['trip_distance'] /
    (clean_df['trip_duration_min'] / 60)
)

In [18]:
clean_df['speed_exception'] = clean_df['avg_speed_mph'] > 80

In [19]:
fare_limit = clean_df['fare_amount'].quantile(0.99)

clean_df['high_fare_exception'] = (
    clean_df['fare_amount'] > fare_limit
)

In [20]:
clean_df['investigation_flag'] = (
    clean_df['speed_exception'] |
    clean_df['high_fare_exception']
)

print(clean_df['investigation_flag'].value_counts())

investigation_flag
False    3295482
True       35181
Name: count, dtype: int64


In [21]:
clean_df['pickup_hour'] = (
    clean_df['tpep_pickup_datetime'].dt.hour
)

cases = clean_df[clean_df['investigation_flag']]

print("Investigation cases:", len(cases))

print("\nCases by vendor:")
print(cases['VendorID'].value_counts())

print("\nCases by hour:")
print(cases['pickup_hour'].value_counts().head(10))

print("\nCases by pickup location:")
print(cases['PULocationID'].value_counts().head(10))

Investigation cases: 35181

Cases by vendor:
VendorID
2    28378
1     5473
7     1196
6      134
Name: count, dtype: int64

Cases by hour:
pickup_hour
15    2923
16    2814
14    2550
17    2422
13    2131
18    1913
19    1715
20    1670
12    1650
22    1635
Name: count, dtype: int64

Cases by pickup location:
PULocationID
132    15169
230     1710
138     1662
161      997
163      740
265      648
48       605
164      585
162      574
68       565
Name: count, dtype: int64


In [22]:
print("Total records:", len(df))
print("Clean records:", len(clean_df))
print("Investigation cases:", len(cases))
print("Speed exceptions:", clean_df['speed_exception'].sum())
print("High-fare exceptions:", clean_df['high_fare_exception'].sum())

Total records: 3475226
Clean records: 3330663
Investigation cases: 35181
Speed exceptions: 2181
High-fare exceptions: 33100
